# 05 · Normalize masters and sales orders

Run this notebook from the `silver/` folder with the existing project kernel. It selects and verifies Bronze manifests, then rebuilds the required prior stages in memory. Reuse the same `silver_settings.json` to review a consistent selection across notebooks.

In [1]:
from IPython.display import display
import pandas as pd

from silver_notebook import build, frame, input_frame, settings_from_file

model = build("orders", settings_from_file())
display(input_frame(model))

,source,manifest
0,erpnext,batch/erpnext/manual__2026-11-21T15%3A27%3A44....
1,dealer-sql,batch/dealer-sql/manual__2026-11-21T15%3A28%3A...
2,parent-pos,batch/parent-pos/manual__2026-11-21T15%3A27%3A...
3,dealer_export,files/otc-inbound/dealer/exports/dealer-export...
4,parent_finance,files/otc-inbound/parent/finance/dealer-order-...
5,dealer_finance,files/otc-inbound/dealer/finance/dealer-order-...
6,mdm,files/otc-inbound/seed-v1/8fab9f89f5242c6819f2...


## Declare the grain and inspect the keys

| Entity | One row represents |
|---|---|
| Company | An operating legal entity |
| Source Customer and Source Product | A native account or sellable item in its source scope |
| Warehouse and GL Account | A source location or account inside a company |
| Sales Order | A source order header |
| Sales Order Line | A distinct native item row on a source order |

ERPNext and the dealer use different order representations. The header points to a customer. The line points to a product and retains ordered quantity, UOM and recorded amount. The parent fixture alias P-S1 is for lookup. The actual ERPNext document ID is used in the key. Dealer amounts are converted from INR paise using decimal arithmetic. A missing unit price stays empty.

In [2]:
display(pd.DataFrame([
    {"table": table, "rows": len(rows)} for table, rows in model.tables.items()
]))
display(frame(model, "source_customer"), frame(model, "source_product"))

,table,rows
0,legal_entity,2
1,source_customer,10
2,source_product,10
3,warehouse,8
4,gl_account,100
5,reference_code,239
6,sales_order,12
7,sales_order_line,12


,id,legal_entity_id,source_id,name,registration,fixture_alias
0,"[""parent"",""OTC P-C1 Metro Industrial Services""]",parent,OTC P-C1 Metro Industrial Services,OTC P-C1 Metro Industrial Services,DEMO-100,P-C1
1,"[""parent"",""OTC P-C2 Pioneer Engineering""]",parent,OTC P-C2 Pioneer Engineering,OTC P-C2 Pioneer Engineering,DEMO-200,P-C2
2,"[""parent"",""OTC P-C3 Apex Industrial Supply""]",parent,OTC P-C3 Apex Industrial Supply,OTC P-C3 Apex Industrial Supply,DEMO-300,P-C3
3,"[""parent"",""OTC P-C4 Summit Process Systems""]",parent,OTC P-C4 Summit Process Systems,OTC P-C4 Summit Process Systems,DEMO-400,P-C4
4,"[""parent"",""OTC P-C5 Harbor Utilities""]",parent,OTC P-C5 Harbor Utilities,OTC P-C5 Harbor Utilities,DEMO-500,P-C5
5,"[""dealer"",""D-C1""]",dealer,D-C1,Metro Ind. Services,DEMO-100,D-C1
6,"[""dealer"",""D-C2""]",dealer,D-C2,Pioneer Engineering,DEMO-200,D-C2
7,"[""dealer"",""D-C3""]",dealer,D-C3,Apex Industrial,None,D-C3
8,"[""dealer"",""D-C4""]",dealer,D-C4,Summit Process,None,D-C4
9,"[""dealer"",""D-C5""]",dealer,D-C5,Valley Fabrication,DEMO-600,D-C5


,id,legal_entity_id,source_id,name,uom,manufacturer,part
0,"[""parent"",""P-FILT""]",parent,P-FILT,OTC Filter,Nos,None,None
1,"[""parent"",""P-GAUGE""]",parent,P-GAUGE,OTC Pressure gauge,Nos,None,None
2,"[""parent"",""P-PUMP""]",parent,P-PUMP,OTC Replacement pump,Nos,None,None
3,"[""parent"",""P-SEAL""]",parent,P-SEAL,OTC Seal,Nos,None,None
4,"[""parent"",""P-VALVE""]",parent,P-VALVE,OTC Valve,Nos,None,None
5,"[""dealer"",""D-FLT""]",dealer,D-FLT,Filter,Each,None,F200
6,"[""dealer"",""D-HOSE""]",dealer,D-HOSE,Hydraulic hose,Each,ACME,H900
7,"[""dealer"",""D-PUMP""]",dealer,D-PUMP,Replacement pump,Each,ACME,RP100
8,"[""dealer"",""D-SEAL""]",dealer,D-SEAL,Seal,Pack,ACME,S20
9,"[""dealer"",""D-VALVE""]",dealer,D-VALVE,Valve,Each,ACME,V50


In [3]:
orders = frame(model, "sales_order")
lines = frame(model, "sales_order_line")
if not orders.empty and not lines.empty:
    selected = orders.loc[orders["order_reference"].isin(["P-S1", "D-S1"])]
    display(selected)
    display(lines.loc[lines["sales_order_id"].isin(selected["id"])])
else:
    display(orders, lines)
display(pd.DataFrame(model.exceptions))

,id,legal_entity_id,source_id,order_reference,customer_source_id,source_customer_id,order_date,currency,source_status,status_code,docstatus
0,"[""parent"",""SAL-ORD-2026-00001""]",parent,SAL-ORD-2026-00001,P-S1,OTC P-C1 Metro Industrial Services,"[""parent"",""OTC P-C1 Metro Industrial Services""]",2026-11-02,INR,Completed,completed,1.0
7,"[""dealer"",""D-S1""]",dealer,D-S1,D-S1,D-C1,"[""dealer"",""D-C1""]",2026-11-05,INR,None,None,NaN


,id,legal_entity_id,source_id,sales_order_id,product_source_id,source_product_id,planned_warehouse_id,quantity,uom,uom_basis,unit_price,amount
0,"[""parent"",""SAL-ORD-2026-00001"",""vqgdj6mnv1""]",parent,vqgdj6mnv1,"[""parent"",""SAL-ORD-2026-00001""]",P-PUMP,"[""parent"",""P-PUMP""]","[""parent"",""P-WH1 - OPD""]",10.0,Nos,transaction,10000.0,100000.0
7,"[""dealer"",""D-S1"",""D-L1""]",dealer,D-L1,"[""dealer"",""D-S1""]",D-PUMP,"[""dealer"",""D-PUMP""]",None,1,Each,dealer fixture uses master unit,None,40000


""
